# H-N60 → H-N70 small-lesion-aware sampling

H-N60(Dataset001, 1채널) 종료 후 H-N70(Dataset003, 2채널)을 순차 실행합니다. 노트북 커널이 종료돼도 백그라운드 프로세스는 계속 실행됩니다.


In [1]:
from pathlib import Path
import json, os, subprocess, sys, time
import pandas as pd
from IPython.display import clear_output, display

ROOT = Path("/home/stu03/workspace/model")
DATA_ROOT = Path("/home/stu03/workspace/datasets/hemorrhage/v1/model_ready/nnunet")
PYTHON = ROOT / ".venv/bin/python"
BATCH_DIR = ROOT / "runs/batches/H-N60_H-N70"
BATCH_STATUS = BATCH_DIR / "batch_status.json"
assert PYTHON.is_file() and DATA_ROOT.is_dir()


## 1. 설정·테스트·plan 확인


In [2]:
subprocess.run([str(PYTHON), "-m", "pytest", "-q", "tests/test_small_lesion_sampling.py"], cwd=ROOT, check=True)
for name in ("H-N60", "H-N70"):
    subprocess.run([str(PYTHON), "-c", f"from brainon_ai.nnunet.config import load_config; load_config('configs/experiments/hemorrhage/{name}.yaml'); print('{name}: config OK')"], cwd=ROOT, check=True)
subprocess.run([str(PYTHON), "scripts/preprocessing/prepare_small_lesion_plans.py", "--preprocessed-root", str(DATA_ROOT / "nnUNet_preprocessed")], cwd=ROOT, check=True)
subprocess.run([str(PYTHON), "scripts/training/run_h_n60_h_n70.py", "--data-root", str(DATA_ROOT), "--dry-run", "--no-validation"], cwd=ROOT, check=True)


..                                                                       [100%]
2 passed in 0.04s
H-N60: config OK
H-N70: config OK
[ok] H-N60: /home/stu03/workspace/datasets/hemorrhage/v1/model_ready/nnunet/nnUNet_preprocessed/Dataset001_ICHBinary/ICHPlansBaselineSmallLesionV1.json
[ok] H-N70: /home/stu03/workspace/datasets/hemorrhage/v1/model_ready/nnunet/nnUNet_preprocessed/Dataset003_ICHBinary2Window/ICHPlansTwoWindowSmallLesionV1.json
/home/stu03/workspace/model/.venv/bin/python /home/stu03/workspace/model/scripts/preprocessing/prepare_small_lesion_plans.py --preprocessed-root /home/stu03/workspace/datasets/hemorrhage/v1/model_ready/nnunet/nnUNet_preprocessed
H-N60: /home/stu03/workspace/model/.venv/bin/python -m brainon_ai.nnunet.runner --config /home/stu03/workspace/model/runs/batches/H-N60_H-N70/resolved/H-N60.yaml --data-root /home/stu03/workspace/datasets/hemorrhage/v1/model_ready/nnunet --results-root /home/stu03/workspace/model/runs/nnunet --no-validation
H-N70: /home/stu03

CompletedProcess(args=['/home/stu03/workspace/model/.venv/bin/python', 'scripts/training/run_h_n60_h_n70.py', '--data-root', '/home/stu03/workspace/datasets/hemorrhage/v1/model_ready/nnunet', '--dry-run', '--no-validation'], returncode=0)

## 2. 두 실험 백그라운드 순차 실행

처음 실행은 `RESUME=False`. 중단된 체크포인트에서 재개할 때만 `True`로 바꿉니다. 밤에는 H-N60/H-N70 학습을 우선 완료하도록 full-volume validation은 생략하며, 학습 종료 후 별도로 실행합니다.


In [3]:
RESUME = False
GPU = "0"
BATCH_DIR.mkdir(parents=True, exist_ok=True)
command = [str(PYTHON), "scripts/training/run_h_n60_h_n70.py", "--data-root", str(DATA_ROOT), "--gpu", GPU, "--no-validation"]
if RESUME:
    command.append("--resume")
launch_log = (BATCH_DIR / "launcher.log").open("a", encoding="utf-8")
process = subprocess.Popen(command, cwd=ROOT, stdout=launch_log, stderr=subprocess.STDOUT, start_new_session=True)
(BATCH_DIR / "launcher.pid").write_text(str(process.pid), encoding="utf-8")
print("PID:", process.pid)
print("log:", BATCH_DIR / "launcher.log")


PID: 1048376
log: /home/stu03/workspace/model/runs/batches/H-N60_H-N70/launcher.log


## 3. 실시간 진행률


In [4]:
PROGRESS_FILES = {
    "H-N60": ROOT / "runs/nnunet/Dataset001_ICHBinary/nnUNetTrainerICHBaseline__ICHPlansBaselineSmallLesionV1__3d_fullres/fold_0/progress.json",
    "H-N70": ROOT / "runs/nnunet/Dataset003_ICHBinary2Window/nnUNetTrainerICHBaseline__ICHPlansTwoWindowSmallLesionV1__3d_fullres/fold_0/progress.json",
}
def read_json(path):
    try: return json.loads(path.read_text(encoding="utf-8"))
    except (FileNotFoundError, json.JSONDecodeError): return {}
def fmt(seconds):
    if seconds is None: return "-"
    seconds=int(seconds); h, r=divmod(seconds,3600); m,s=divmod(r,60); return f"{h:02d}:{m:02d}:{s:02d}"
while True:
    clear_output(wait=True)
    batch=read_json(BATCH_STATUS); items=batch.get("experiments",{}); rows=[]
    for name,path in PROGRESS_FILES.items():
        p=read_json(path); b=items.get(name,{})
        rows.append({"실험":name,"상태":p.get("status",b.get("status","waiting")),
                     "Epoch":f"{p.get('current_epoch',0)}/{p.get('total_epochs','-')}",
                     "진행률(%)":p.get("progress_percent",0),"ETA":fmt(p.get("eta_seconds")),
                     "Dice":p.get("pseudo_dice"),"Recall":p.get("pseudo_recall"),
                     "Best Composite":p.get("best_composite")})
    print("전체:",batch.get("status","starting"),"현재:",batch.get("current","-")); display(pd.DataFrame(rows))
    if batch.get("status") in {"completed","failed"}: break
    time.sleep(10)


전체: completed 현재: None


,실험,상태,Epoch,진행률(%),ETA,Dice,Recall,Best Composite
0,H-N60,early_stopped,229/300,76.33,00:35:11,None,None,0.824661
1,H-N70,early_stopped,229/300,76.33,00:22:01,None,None,0.826062
